DEVELOPMENT SUMMARY
===================
Competition : GCI NFL Draft Prediction
Final AUC   : 0.85440
Final Rank  : Top 3% (110/3600)
Submissions : 150+

KEY IMPROVEMENTS:

0.84151 → Baseline ensemble

0.84647 → Pure CatBoost discovery  

0.84800 → Feature engineering

0.85043 → Crossed 0.85 barrier

0.85257 → Stacking implementation

0.85440 → Rank averaging final

MY APPROACH:
- One change at a time methodology
- Systematic validation every step
- Healthy model (overfit gap=0.013)
- MICE imputation critical discovery

MY CONTEXT:
- First year undergraduate student
- 5 university exams simultaneously
- Self-directed learning throughout
- Preparing the Final assignment parallely

## Assumptions made before running the code
1. The train and test csv files are present
2. The latest packages are installed

This code was written in VSC 

#### The following code was submitted after multiple revisions and updating my previous codes.
Thank You

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata


In [ ]:

# 1. Loaing the datasets
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")
submission = sample_submission.copy()

# 2. Making new features for models
for df in [train, test]:
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Speed_Score'] = df['Weight'] / df['Sprint_40yd']
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump']
    df['Power_Index'] = df['Vertical_Jump'] * df['Weight']
    df['Total_Agility'] = df['Agility_3cone'] + df['Shuttle']
    df['Agility_Ratio'] = df['Agility_3cone'] / df['Shuttle']
    df['Strength_Index'] = df['Bench_Press_Reps'] / df['Weight']
    df['Speed_pos_type_diff'] = df['Sprint_40yd'] - df.groupby('Position_Type')['Sprint_40yd'].transform('mean')
    for col in ['Agility_3cone', 'Shuttle', 'Bench_Press_Reps']:
        df[f'{col}_missing'] = df[col].isnull().astype(int)
    df['Catch_Radius'] = df['Height'] + (df['Vertical_Jump']/100) + (df['Broad_Jump']/100) - (df['Shuttle']/10)
    df['Height_Adj_Speed'] = (df['Weight'] * (df['Height'] ** 0.5)) / (df['Sprint_40yd'] ** 4)
    for stat in ['Sprint_40yd', 'BMI', 'Catch_Radius']:
        df[f'{stat}_pos_diff'] = df[stat] - df.groupby('Position')[stat].transform('mean')
    df['Jump_Rank'] = df.groupby('Position')['Vertical_Jump'].rank(ascending=False)
    df['Speed_Rank'] = df.groupby('Position')['Sprint_40yd'].rank(ascending=True)
    df['Combined_Rank'] = df['Jump_Rank'] + df['Speed_Rank']
    df['Size_Agility'] = df['BMI'] * df['Total_Agility']
    df['Peak_Power_Watts'] = (60.7 * (df['Vertical_Jump'] * 100)) + (45.3 * df['Weight']) - 2055
    df['Total_Missing_Drills'] = df[['Agility_3cone_missing', 'Shuttle_missing', 'Bench_Press_Reps_missing']].sum(axis=1)

    df['Is_Young_Prospect'] = (df['Age'] <= 21).astype(int)
    df['Is_Veteran_Prospect'] = (df['Age'] >= 24).astype(int)
    df['Speed_Percentile'] = df.groupby('Position')['Sprint_40yd'].rank(pct=True, ascending=True)
    df['Jump_Percentile'] = df.groupby('Position')['Vertical_Jump'].rank(pct=True, ascending=False)
# 3. Doing the preprocessing
school_freq = train['School'].value_counts().to_dict()
train['School_Freq'] = train['School'].map(school_freq)
test['School_Freq'] = test['School'].map(school_freq).fillna(0)

train = train.drop(columns=["Id", "School"])
test = test.drop(columns=["Id", "School"])

le = LabelEncoder()
for col in ["Player_Type", "Position_Type", "Position"]:
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

X = train.drop(columns=["Drafted"])
y = train["Drafted"]

imputer = IterativeImputer(random_state=2025)
X_imputed = pd.DataFrame(
    imputer.fit_transform(X), columns=X.columns
)
test_imputed = pd.DataFrame(
    imputer.transform(test), columns=test.columns
)

# 4.    MODEL DEFINITIONS
cat_model = CatBoostClassifier(
    iterations=250, depth=7, learning_rate=0.05,
    subsample=0.9, bootstrap_type='Bernoulli',
    random_seed=2025, verbose=0
)
xgb_model = XGBClassifier(
    n_estimators=150, max_depth=5, learning_rate=0.05,
    random_state=2025, eval_metric='logloss'
)
lgbm_model = LGBMClassifier(
    n_estimators=200, learning_rate=0.05,
    random_state=2025, objective='binary'
)

# 5. CV
rskf = RepeatedStratifiedKFold(
    n_splits=20, n_repeats=3, random_state=42
)
oof_cat  = np.zeros(len(X_imputed))
oof_xgb  = np.zeros(len(X_imputed))
oof_lgbm = np.zeros(len(X_imputed))
test_cat  = np.zeros(len(test_imputed))
test_xgb  = np.zeros(len(test_imputed))
test_lgbm = np.zeros(len(test_imputed))

# 7. loop through the folds
for fold, (train_idx, valid_idx) in enumerate(
    rskf.split(X_imputed, y)):

    X_t = X_imputed.iloc[train_idx]
    X_v = X_imputed.iloc[valid_idx]
    y_t = y.iloc[train_idx]

    X_t = X_t.replace([np.inf, -np.inf], np.nan)
    X_v = X_v.replace([np.inf, -np.inf], np.nan)
    test_imputed = test_imputed.replace([np.inf, -np.inf], np.nan)

    fold_median = X_t.median()
    X_t = X_t.fillna(fold_median)
    X_v = X_v.fillna(fold_median)
    test_imputed = test_imputed.fillna(fold_median)

    numeric_cols = X_t.select_dtypes(include=[np.number]).columns
    X_t[numeric_cols] = X_t[numeric_cols].clip(-1e8, 1e8)
    X_v[numeric_cols] = X_v[numeric_cols].clip(-1e8, 1e8)
    test_imputed[numeric_cols] = test_imputed[numeric_cols].clip(-1e8, 1e8)

    cat_model.fit(X_t, y_t)
    xgb_model.fit(X_t, y_t)
    lgbm_model.fit(X_t, y_t)

    oof_cat[valid_idx]  = cat_model.predict_proba(X_v)[:, 1]
    oof_xgb[valid_idx]  = xgb_model.predict_proba(X_v)[:, 1]
    oof_lgbm[valid_idx] = lgbm_model.predict_proba(X_v)[:, 1]

    test_cat  += cat_model.predict_proba(test_imputed)[:, 1] / (20 * 3)
    test_xgb  += xgb_model.predict_proba(test_imputed)[:, 1] / (20 * 3)
    test_lgbm += lgbm_model.predict_proba(test_imputed)[:, 1] / (20 * 3)

# 8. avg the ranks of the predictions
rank_cat  = rankdata(test_cat)  / len(test_cat)
rank_xgb  = rankdata(test_xgb)  / len(test_xgb)
rank_lgbm = rankdata(test_lgbm) / len(test_lgbm)

final_preds = (rank_cat + rank_xgb + rank_lgbm) / 3
submission["Drafted"] = final_preds
submission.to_csv("submission.csv", index=False)

c:\PES\GCI\Python\.venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


XGBoostError: [12:57:53] C:\actions-runner\_work\xgboost\xgboost\src\data\gradient_index.h:99: Check failed: valid: Input data contains `inf` or a value too large, while `missing` is not set to `inf`